# MSTAR SAR Target Recognition - Quickstart & Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohitgit1/Target-Detection-in-MSTAR-Images/blob/main/notebooks/01_quickstart_and_inference.ipynb)

This notebook demonstrates how to load single-channel Synthetic Aperture Radar (SAR) chips from the MSTAR dataset, apply radar-specific speckle filtering and dynamic range compression, classify targets using **A-ConvNet**, and generate **Grad-CAM** radar attention maps.

In [ ]:
# Install / setup dependencies if running in Google Colab
import sys
!pip install -q torch torchvision scikit-learn matplotlib pillow
sys.path.append('..')

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from mstar_atr.constants import CLASSES, TARGET_METADATA, DEFAULT_CROP_SIZE
from mstar_atr.data.dataset import SARTransform
from mstar_atr.filters.speckle import lee_filter, frost_filter, amplitude_to_db, normalize_image
from mstar_atr.models.aconvnet import AConvNet
from mstar_atr.interpretation.gradcam import GradCAM, overlay_gradcam_on_sar

## 1. Load a Sample SAR Target Chip
Let's load a sample MSTAR radar return (e.g. BMP-2 Infantry Fighting Vehicle).

In [ ]:
sample_path = '../assets/samples/BMP2_sample.jpeg'
if not os.path.exists(sample_path):
    from mstar_atr.data.dataset import create_synthetic_mstar_dataset
    create_synthetic_mstar_dataset('sample_data', samples_per_class=1)
    sample_path = 'sample_data/train/BMP2/BMP2_train_000.jpeg'

raw_img = Image.open(sample_path).convert('L')
raw_arr = np.array(raw_img, dtype=np.float32)
print(f'Loaded chip shape: {raw_arr.shape}, dynamic range: [{raw_arr.min():.1f}, {raw_arr.max():.1f}]')

## 2. SAR Preprocessing & Speckle Reduction
Comparing raw amplitude, dynamic range logarithmic compression (dB), and adaptive Lee speckle filtering.

In [ ]:
db_img = amplitude_to_db(raw_arr)
lee_filtered = lee_filter(raw_arr, window_size=5)
lee_db = amplitude_to_db(lee_filtered)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_arr, cmap='gray')
axes[0].set_title('Raw SAR Amplitude')
axes[0].axis('off')

axes[1].imshow(db_img, cmap='gray')
axes[1].set_title('Dynamic Range (dB Scale)')
axes[1].axis('off')

axes[2].imshow(lee_db, cmap='gray')
axes[2].set_title('Adaptive Lee Filter (5x5) + dB')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 3. Model Inference & Classification
Instantiate A-ConvNet and predict target class probabilities.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AConvNet(num_classes=len(CLASSES), in_channels=1).to(device)
model.eval()

transform = SARTransform(output_size=DEFAULT_CROP_SIZE, is_training=False)
input_tensor = transform(raw_arr).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)
    probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()

pred_idx = int(np.argmax(probs))
pred_class = CLASSES[pred_idx]
meta = TARGET_METADATA.get(pred_class, {})

print(f'Predicted Class: {pred_class} ({meta.get("full_name", "")})')
print(f'Category:        {meta.get("category", "")}')
print(f'Confidence:      {probs[pred_idx]*100:.2f}%')

## 4. Grad-CAM Radar Interpretability
Visualize which radar dihedral reflections and shadow geometry drove the neural network classification decision.

In [ ]:
cam_engine = GradCAM(model)
heatmap, _, _ = cam_engine.generate_heatmap(input_tensor, target_class=pred_idx)
overlay = overlay_gradcam_on_sar(raw_arr, heatmap, alpha=0.5, colormap_name='inferno')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw_arr, cmap='gray')
axes[0].set_title(f'Raw Target: {pred_class}')
axes[0].axis('off')

axes[1].imshow(overlay)
axes[1].set_title(f'Grad-CAM Overlay ({pred_class})')
axes[1].axis('off')

plt.tight_layout()
plt.show()